# Federated Optimization: FedProx, SCAFFOLD, and FedNova

FedAvg converges poorly when client data distributions are heterogeneous. This notebook covers algorithms designed to address the client drift problem.

## Why FedAvg Fails on Non-IID Data

During local training, each client's model moves toward the local optimum. After many local epochs, local models can diverge significantly from the global model. When averaged, the result may be far from any individual client's optimum and far from the true global optimum.

**Client drift** is measurable: the gap between where FedAvg converges and the centralized training optimum grows with the number of local steps and the degree of data heterogeneity.

In [ ]:
from dataclasses import dataclass, field
from typing import Callable

@dataclass
class FedProxConfig:
    mu: float = 0.01       # proximal term weight
    local_epochs: int = 5
    lr: float = 0.01
    n_rounds: int = 20
    fraction: float = 0.5

class FedProxSimulator:
    def __init__(self, clients: list, global_model: list, config: FedProxConfig):
        self.clients = clients
        self.global_model = list(global_model)
        self.config = config
        self.history = []
        self.rng = random.Random(42)

    def _local_train_prox(self, client, global_model: list) -> tuple:
        model = list(global_model)
        total_loss = 0.0
        for _ in range(self.config.local_epochs):
            batch = [self.rng.gauss(client['mean'], 1.0) for _ in range(32)]
            # Standard gradient
            grad = sum(2*(model[0]-x) for x in batch) / len(batch)
            # FedProx proximal term: pull toward global model
            prox_grad = self.config.mu * (model[0] - global_model[0])
            model[0] -= self.config.lr * (grad + prox_grad)
            total_loss += sum((model[0]-x)**2 for x in batch) / len(batch)
        return model, total_loss / self.config.local_epochs

    def run(self) -> list:
        for r in range(self.config.n_rounds):
            n_sel = max(1, int(len(self.clients)*self.config.fraction))
            selected = self.rng.sample(self.clients, n_sel)
            updates, weights, losses = [], [], []
            for c in selected:
                upd, loss = self._local_train_prox(c, self.global_model)
                updates.append(upd)
                weights.append(c['n'])
                losses.append(loss)
            total_w = sum(weights)
            self.global_model = [sum(u[i]*w for u,w in zip(updates,weights))/total_w
                                  for i in range(len(self.global_model))]
            self.history.append({'round': r+1, 'model': self.global_model[0],
                                  'loss': sum(losses)/len(losses)})
        return self.history

# Non-IID clients: data means spread across 1-10
clients = [{'id': i, 'n': rng.randint(100,500), 'mean': float(i+1)} for i in range(10)]
true_global_mean = 5.5

# FedAvg baseline
class FedAvgBaseline:
    def __init__(self, clients, global_model, lr=0.01, epochs=5, rounds=20, frac=0.5):
        self.clients = clients; self.model = list(global_model)
        self.lr=lr; self.epochs=epochs; self.rounds=rounds; self.frac=frac
        self.history=[]; self.rng=random.Random(42)
    def run(self):
        for r in range(self.rounds):
            n_sel = max(1, int(len(self.clients)*self.frac))
            sel = self.rng.sample(self.clients, n_sel)
            updates, weights = [], []
            for c in sel:
                m = list(self.model)
                for _ in range(self.epochs):
                    batch=[self.rng.gauss(c['mean'],1.0) for _ in range(32)]
                    grad=sum(2*(m[0]-x) for x in batch)/32
                    m[0]-=self.lr*grad
                updates.append(m); weights.append(c['n'])
            tw=sum(weights)
            self.model=[sum(u[i]*w for u,w in zip(updates,weights))/tw for i in range(len(self.model))]
            self.history.append({'round':r+1,'model':self.model[0]})
        return self.history

fedavg = FedAvgBaseline(clients, [0.0])
fedprox = FedProxSimulator(clients, [0.0], FedProxConfig(mu=0.1))
fa_hist = fedavg.run()
fp_hist = fedprox.run()

print(f'True global mean: {true_global_mean}')
print(f'{'Round':>7} {'FedAvg':>10} {'FedProx':>10} {'FedAvg Err':>12} {'FedProx Err':>13}')
print('-' * 55)
for r in [4, 9, 14, 19]:
    fa_m = fa_hist[r]['model']; fp_m = fp_hist[r]['model']
    print(f"{r+1:>7} {fa_m:>10.4f} {fp_m:>10.4f} {abs(fa_m-true_global_mean):>12.4f} {abs(fp_m-true_global_mean):>13.4f}")

## SCAFFOLD and Variance Reduction

SCAFFOLD (Karimireddy et al. 2020) corrects client drift using control variates:
- Each client maintains a correction vector (the difference between its local gradient and the global gradient)
- During local training, the correction vector is subtracted from the local gradient, removing the drift
- The correction is updated after each round

SCAFFOLD converges provably faster than FedProx on heterogeneous data at the cost of storing and communicating control variates (2x communication overhead).

**FedNova** normalizes local updates by the number of local steps before aggregation, correcting for the fact that clients with different dataset sizes take different numbers of steps per round.